# Lab 1.1, Build 1: Triage Tina's Failures

**Before you start:** select **Cell > Run All** to initialize the harness.

Tina answered the same compliance question three times. Each answer is wrong in a different way.
Read each response, run all three endpoints, then record your diagnosis.

Cells marked `# ── YOUR WORK ──` are the ones you edit.

In [ ]:
# ── Harness setup (run once) ──────────────────────────────────────────────────
import sys, os, json, pathlib, time

sys.path.insert(0, '/opt/ara/lib')
from tina.client import llm_client, model_fast, model_strong

env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

client = llm_client()
FAST   = model_fast()
STRONG = model_strong()
TRACES = pathlib.Path('/home/elastic/.traces')
TRACES.mkdir(parents=True, exist_ok=True)
RESULTS = pathlib.Path('/opt/ara/results')
DEV_QUESTIONS = list(pathlib.Path('/home/elastic/dev-sets').glob('*.jsonl'))

print(f'Harness ready. FAST={FAST}, STRONG={STRONG}')

---
## Build 1: Triage

Tina answered the same compliance question three times. Each answer is wrong in a different way.
Read each one, run all three endpoints yourself on the same question, then record your diagnosis.

In [ ]:
# ── The three broken responses (do not edit) ─────────────────────────────────
QUESTION = 'What is Cortex Bank\'s CTR threshold for cash transactions, and is a $9,500 deposit reportable?'

RESPONSE_1 = {
    'id': 'response_1',
    'text': 'The CTR threshold is $5,000 for cash transactions. A $9,500 deposit would be reportable.'
}

RESPONSE_2 = {
    'id': 'response_2',
    'text': '[0.023, -0.451, 0.887, 0.112, -0.334, 0.556, ...]  (768-dimensional vector)'
}

RESPONSE_3 = {
    'id': 'response_3',
    'answer': 'The CTR filing deadline is 45 days from the triggering transaction.',
    'policy_id': 'policy-011',
    'confidence': 'high'
}

for r in [RESPONSE_1, RESPONSE_2, RESPONSE_3]:
    print(f"\n--- {r['id']} ---")
    print(json.dumps(r, indent=2))

In [ ]:
# ── Run all three endpoint types on the question ──────────────────────────────
# This cell demonstrates what each endpoint type actually returns.
# The LLM generates text; the embedding model returns a vector;
# the reranker scores a pair of (query, document).

print('=== LLM endpoint ===')
llm_resp = client.chat.completions.create(
    model=FAST,
    messages=[{'role': 'user', 'content': QUESTION}],
    temperature=0,
)
print(llm_resp.choices[0].message.content[:200])

print('\n=== Embedding endpoint (returns a vector, not text) ===')
# Jina embeddings run on Elastic's inference service, not the AI gateway proxy.
from elasticsearch import Elasticsearch as _ES
_es = _ES(os.environ['ES_URL'], api_key=os.environ['ES_API_KEY'])
embed_id = os.environ.get('ARA_EMBED_ID', '.jina-embeddings-v5-text-small')
embed_resp = _es.inference.inference(
    inference_id=embed_id,
    task_type='text_embedding',
    body={'input': [QUESTION]},
)
vec = embed_resp['text_embedding'][0]['embedding']
print(f'Vector of {len(vec)} floats: {vec[:5]}...')

print('\n=== Reranker endpoint (scores a query-document pair) ===')
# Rerankers don't generate answers; they return a relevance score.
print('Reranker returns a relevance score between 0 and 1, not an answer.')
print('Asking it to answer a question directly produces no meaningful output.')

In [ ]:
# ── YOUR WORK ── Diagnose each response ─────────────────────────────────────
# For each response, fill in:
#   failure_mode: 'hallucination' | 'wrong_model_type' | 'ungrounded'
#   model_type:   'llm' | 'embedding_model' | 'reranker'

diagnoses = [
    {'failure_mode': '',  'model_type': ''},  # ← Response 1
    {'failure_mode': '',  'model_type': ''},  # ← Response 2
    {'failure_mode': '',  'model_type': ''},  # ← Response 3
]

In [ ]:
# Record diagnoses
assert all(d['failure_mode'] and d['model_type'] for d in diagnoses), \
    'Fill in failure_mode and model_type for all three responses.'
results = {
    'diagnoses': diagnoses,
    'endpoint_calls': ['llm', 'embedding_model', 'reranker'],
    'timestamp': time.time()
}
(TRACES / 'diagnose-results.json').write_text(json.dumps(results, indent=2))
print('Diagnoses recorded. Select Check in the sidebar.')